In [ ]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import textwrap
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage


load_dotenv()


In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")
QDRANT_PORT = 6334 
COLLECTION_NAME = "Crail_data"

In [ ]:
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0.5,
    max_tokens=5000,
    top_p=0.95,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None,
)

In [ ]:
qdrant = QdrantClient("http://localhost", port=QDRANT_PORT)  # Use your secondary container port
embedding_model  = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
def search_qdrant(user_query, top_k=5):
    query_vector = embedding_model.encode(user_query).tolist()
    results = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=top_k
    )
    return results

In [ ]:
def ask_llm(question, context):
    prompt = f"""
You are a helpful hospital assistant.

Use the following context to answer the user's question.

Context:
{textwrap.indent(context, '  ')}

Question:
{question}

Answer:"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

In [ ]:
def query_system(user_query):
    print(f"\n🔍 User Question: {user_query}\n")

    results = search_qdrant(user_query)
    context_chunks = []

    for res in results:
        payload = res.payload.get("original_data")
        if payload:
            context_chunks.append(str(payload))

    if not context_chunks:
        return "No relevant documents found in the database."

    combined_context = "\n\n".join(context_chunks)
    answer = ask_llm(user_query, combined_context)

    print("\n💬 LLM Response:")
    print(textwrap.fill(answer, width=100))

In [ ]:
"Who is the most experienced cardiologist?"

"What treatments are available for heart-related issues?"

"Can I consult Dr. Rajiv Kumar online?"

"How much does knee replacement cost?"

In [ ]:
question = "Dr. Anjali Mehta's consultation fee and available timings on Fridays."
query_system(question)